In [1]:
%load_ext autoreload
%autoreload 2

import tinytensor as tt
import numpy as np


class Layer:
    def __init__(self, ins: int, outs: int, rng):
        self.weights = tt.Tensor(rng.normal(0, np.sqrt(2 / ins), (outs, ins)), True)
        self.bias = tt.Tensor(np.zeros((outs,)), True)

        def _forward(data: tt.Tensor):
            return tt.relu(self.weights @ data + self.bias)

        self._forward = _forward

    def forward(self, data: tt.Tensor):
        return self._forward(data)

    def update(self, factor: float):
        self.weights.data -= factor * self.weights.grad
        self.bias.data -= factor * self.bias.grad

    def zeroGrad(self):
        self.weights.grad *= 0
        self.bias.grad *= 0

    def outLayer(self):
        def _forward(data: tt.Tensor):
            return self.weights @ data + self.bias

        self._forward = _forward
        return self


class MLP:
    def __init__(self, ins: int, hidden: list[int], outs: int, seed:int = 71116):
        rng = np.random.default_rng(seed)
        self.ins = ins
        self.shape = hidden
        self.layers = (
            [Layer(ins, hidden[0], rng)]
            + [Layer(a, b, rng) for a, b in zip(hidden, hidden[1:])]
            + [Layer(hidden[-1], outs, rng).outLayer()]
        )

    def forward(self, data: tt.Tensor):
        activation = data
        for layer in self.layers:
            activation = layer.forward(activation)
        return activation

    def update(self, factor: float):
        for layer in self.layers:
            layer.update(factor)

    def zeroGrad(self):
        for layer in self.layers:
            layer.zeroGrad()

In [83]:
NN = MLP(3, [8], 3, seed=2)

# x = tt.Tensor([[1, 2, 3], [1, 1, 1], [-1, 2, 0]])
# y = tt.Tensor([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
x = tt.Tensor([1, 2, 3])
y = tt.Tensor([10, 0, 0])

learning_rate = 0.001
epochs = 100

for i in range(epochs):
    NN.zeroGrad()
    vec_loss = NN.forward(x) - y
    loss = tt.einsum([vec_loss, vec_loss], "i,i->")
    loss.grad = np.array(1.0)
    loss.backward()
    NN.update(learning_rate)

    if i % 10 == 0:
        print(f"Epoch: {i:4d} Total Loss: {loss.data:.6f}")

print(f"predicted: {NN.forward(x).data} target: {y.data}")

Epoch:    0 Total Loss: 58.357043
Epoch:   10 Total Loss: 6.274711
Epoch:   20 Total Loss: 0.388211
Epoch:   30 Total Loss: 0.024875
Epoch:   40 Total Loss: 0.002046
Epoch:   50 Total Loss: 0.000197
Epoch:   60 Total Loss: 0.000020
Epoch:   70 Total Loss: 0.000002
Epoch:   80 Total Loss: 0.000000
Epoch:   90 Total Loss: 0.000000
predicted: [1.00000084e+01 1.00919322e-06 4.91926962e-05] target: [10.  0.  0.]


In [ ]:
NN_demo = MLP(3, [16], 3, seed=2)

train_x = [
    tt.Tensor([3.0, 1.0, 1.0]),
    tt.Tensor([1.0, 3.0, 1.0]),
    tt.Tensor([1.0, 1.0, 3.0]),
    tt.Tensor([2.0, 2.0, 3.0]),
]
train_y = [
    tt.Tensor([1.0, 0.0, 0.0]),
    tt.Tensor([0.0, 1.0, 0.0]),
    tt.Tensor([0.0, 0.0, 1.0]),
    tt.Tensor([0.0, 0.0, 1.0]),
]

lr = 0.01
epochs = 2000

for epoch in range(epochs):
    total_loss = 0.0
    for x, y in zip(train_x, train_y):
        NN_demo.zeroGrad()
        vec_loss = NN_demo.forward(x) - y
        loss = tt.einsum([vec_loss, vec_loss], "i,i->")
        loss.grad = np.array(1.0)
        loss.backward()
        NN_demo.update(lr)
        total_loss += loss.data

    if epoch % 100 == 0:
        print(f"epoch {epoch:4d}  avg loss: {total_loss / len(train_x):.33f}")

print()
for x, y in zip(train_x, train_y):
    print("input:", x.data, " prediction:", NN_demo.forward(x).data, " target:", y.data)

epoch    0  avg loss: 28.514437293760632741168592474423349
epoch  100  avg loss: 0.000091645153356009227880486778250
epoch  200  avg loss: 0.000000077232269351207106635533346
epoch  300  avg loss: 0.000000000083281398614984023123007
epoch  400  avg loss: 0.000000000000125592142703281491318
epoch  500  avg loss: 0.000000000000000236385792778398184
epoch  600  avg loss: 0.000000000000000000488768770229873
epoch  700  avg loss: 0.000000000000000000001043229233892
epoch  800  avg loss: 0.000000000000000000000002248765372
epoch  900  avg loss: 0.000000000000000000000000004905214
epoch 1000  avg loss: 0.000000000000000000000000000010557
epoch 1100  avg loss: 0.000000000000000000000000000000117
epoch 1200  avg loss: 0.000000000000000000000000000000102
epoch 1300  avg loss: 0.000000000000000000000000000000102
epoch 1400  avg loss: 0.000000000000000000000000000000065
epoch 1500  avg loss: 0.000000000000000000000000000000049
epoch 1600  avg loss: 0.000000000000000000000000000000071
epoch 1700  a

In [85]:
test_x = tt.Tensor([3.0, 2.0, 0.0])
test_y = tt.Tensor([1.0, 0.0, 0.0])

print("input:", test_x.data, " prediction:", NN_demo.forward(test_x).data, " target:", test_y.data)

input: [3. 2. 0.]  prediction: [ 1.0948001   0.19484269 -0.0174826 ]  target: [1. 0. 0.]
